In [1]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
import itertools
import time

# MLP Tuning

In [1]:

DATA_DIR = 'processed_data'
BASE_DIR = 'Hybrid Model 2.0'
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

if not os.path.exists(BASE_DIR): os.makedirs(BASE_DIR)
if not os.path.exists(RESULTS_DIR): os.makedirs(RESULTS_DIR)
if not os.path.exists(MODEL_DIR): os.makedirs(MODEL_DIR)

PARAM_GRID = {
    'hidden_layers': [1, 2],           
    'fusion_dim': [64, 128, 256],      
    'dropout': [0.3, 0.5],             
    'lr': [0.001, 0.0005],
    'weight_decay': [1e-4, 1e-3] 
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DynamicFusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, fusion_dim, dropout_rate):
        super(DynamicFusionMLP, self).__init__()
        self.layers = nn.ModuleList()
        
        # Layer 1
        self.layers.append(nn.Linear(input_dim, 256))
        self.layers.append(nn.BatchNorm1d(256)) 
        self.layers.append(nn.ReLU())
        self.layers.append(nn.Dropout(dropout_rate))
        
        current_dim = 256
        
        if hidden_layers == 2:
            self.layers.append(nn.Linear(256, 128))
            self.layers.append(nn.BatchNorm1d(128))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(dropout_rate))
            current_dim = 128
            
        self.fusion_layer = nn.Linear(current_dim, fusion_dim)
        self.fusion_act = nn.ReLU()
        
        self.head = nn.Linear(fusion_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        fused_features = self.fusion_act(self.fusion_layer(x))
        output = self.sigmoid(self.head(fused_features))
        return fused_features, output

def stage1_tune_mlp():
    print(f"\n" + "="*60)
    print(f"STAGE 1: MLP Hyperparameters TUNING (10-FOLD CV)")
    print("="*60)
    
    X_absa = np.load(os.path.join(DATA_DIR, 'X_absa_train.npy'))
    X_emo  = np.load(os.path.join(DATA_DIR, 'X_emo_train.npy'))
    y      = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    groups = np.load(os.path.join(DATA_DIR, 'groups_train.npy'))
    
    X_concat = np.hstack([X_absa, X_emo])
    input_dim = X_concat.shape[1]
    
    keys, values = zip(*PARAM_GRID.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"   Testing {len(combinations)} MLP configurations...")
    print(f"   (10-Fold CV per config = {len(combinations)*10} training runs)")
    
    results_log = []
    best_f1 = 0.0
    best_config = None
    
    start_total = time.time()
    
    for i, config in enumerate(combinations):
        print(f"\n🔹 Config {i+1}/{len(combinations)}: {config}")
        
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        fold_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(cv.split(X_concat, groups)):
            X_train_t = torch.FloatTensor(X_concat[train_idx]).to(device)
            y_train_t = torch.FloatTensor(y[train_idx]).unsqueeze(1).to(device)
            X_val_t   = torch.FloatTensor(X_concat[val_idx]).to(device)
            y_val     = y[val_idx]
            
            model = DynamicFusionMLP(input_dim, config['hidden_layers'], config['fusion_dim'], config['dropout']).to(device)
            optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
            criterion = nn.BCELoss() 
            
            model.train()
            loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
            for epoch in range(25):
                for bX, bY in loader:
                    optimizer.zero_grad()
                    _, pred = model(bX)
                    loss = criterion(pred, bY)
                    loss.backward()
                    optimizer.step()
            
            model.eval()
            with torch.no_grad():
                _, preds_prob = model(X_val_t)
                preds = (preds_prob > 0.5).float().cpu().numpy()
            
            score = f1_score(y_val, preds)
            fold_scores.append(score)
            
        avg_score = np.mean(fold_scores)
        print(f"   ✅ Average F1: {avg_score:.4f}")
        results_log.append({**config, 'f1_score': avg_score})
        
        if avg_score > best_f1:
            best_f1 = avg_score
            best_config = config

    print("\n" + "="*60)
    print(f"WINNING CONFIG FOUND")
    print("="*60)
    print(f"   Config: {best_config}")
    print(f"   Best CV F1: {best_f1:.4f}")
    
    print("\n   Retraining MLP on full Training Data...")
    
    full_X_t = torch.FloatTensor(X_concat).to(device)
    full_y_t = torch.FloatTensor(y).unsqueeze(1).to(device)
    
    final_model = DynamicFusionMLP(
        input_dim, 
        best_config['hidden_layers'], 
        best_config['fusion_dim'], 
        best_config['dropout']
    ).to(device)
    
    optimizer = optim.Adam(final_model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
    criterion = nn.BCELoss()
    
    final_model.train()
    loader = DataLoader(TensorDataset(full_X_t, full_y_t), batch_size=32, shuffle=True)
    
    for epoch in range(30):
        for bX, bY in loader:
            optimizer.zero_grad()
            _, pred = final_model(bX)
            loss = criterion(pred, bY)
            loss.backward()
            optimizer.step()
            
    print(f"   Saving Best Model to {MODEL_DIR}...")
    torch.save(final_model.state_dict(), os.path.join(MODEL_DIR, 'Best_MLP_Stage1.pth'))
    
    pd.DataFrame(results_log).sort_values('f1_score', ascending=False).to_csv(os.path.join(RESULTS_DIR, 'Stage1_MLP_Results.csv'), index=False)
    pd.DataFrame([best_config]).to_csv(os.path.join(RESULTS_DIR, 'best_mlp_config.csv'), index=False)
    
    total_time = (time.time() - start_total) / 60
    print(f"MLP training Complete in {total_time:.1f} minutes.")

if __name__ == "__main__":
    stage1_tune_mlp()


STAGE 1: MLP Hyperparameters TUNING (10-FOLD CV)
   Testing 48 MLP configurations...
   (10-Fold CV per config = 480 training runs)

🔹 Config 1/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001}
   ✅ Average F1: 0.8481

🔹 Config 2/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.001}
   ✅ Average F1: 0.8191

🔹 Config 3/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001}
   ✅ Average F1: 0.8520

🔹 Config 4/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.001}
   ✅ Average F1: 0.8559

🔹 Config 5/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.0001}
   ✅ Average F1: 0.8540

🔹 Config 6/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.5, 'lr': 0.001, 'weight_decay': 0.001}
   ✅ Average F1: 0.8422

🔹 Config 7/48: {'hidden_layers': 1, 'fusion_dim': 64, 'dropout': 0.5, 'lr':

# XGBoost Tuning

In [3]:
#latest
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import pickle

DATA_DIR = 'processed_data'
BASE_DIR = 'Hybrid Model 2.0'
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

if not os.path.exists(BASE_DIR): os.makedirs(BASE_DIR)
if not os.path.exists(RESULTS_DIR): os.makedirs(RESULTS_DIR)
if not os.path.exists(MODEL_DIR): os.makedirs(MODEL_DIR)

XGB_PARAM_GRID = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [1, 2, 3, 5, 7, 9],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'subsample': [0.8, 1.0]
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DynamicFusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, fusion_dim, dropout_rate):
        super(DynamicFusionMLP, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, 256))
        self.layers.append(nn.BatchNorm1d(256))
        self.layers.append(nn.ReLU())
        self.layers.append(nn.Dropout(dropout_rate))
        
        current_dim = 256
        if hidden_layers == 2:
            self.layers.append(nn.Linear(256, 128))
            self.layers.append(nn.BatchNorm1d(128))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(dropout_rate))
            current_dim = 128
            
        self.fusion_layer = nn.Linear(current_dim, fusion_dim)
        self.fusion_act = nn.ReLU()
        self.head = nn.Linear(fusion_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        fused_features = self.fusion_act(self.fusion_layer(x))
        return fused_features 

def stage2_tune_xgboost():
    print(f"\n" + "="*60)
    print(f"STAGE 2: XGBOOST OPTIMIZATION")
    print("="*60)
    
    print("1. Loading Data...")
    try:
        X_absa = np.load(os.path.join(DATA_DIR, 'X_absa_train.npy'))
        X_emo  = np.load(os.path.join(DATA_DIR, 'X_emo_train.npy'))
        y      = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
        groups = np.load(os.path.join(DATA_DIR, 'groups_train.npy'))
    except FileNotFoundError:
        print("❌ Error: 'processed_data' files not found.")
        return
        
    X_concat = np.hstack([X_absa, X_emo])
    
    print("2. Loading Best MLP Config...")
    try:
        config_df = pd.read_csv(os.path.join(RESULTS_DIR, 'best_mlp_config.csv'))
        best_config = config_df.iloc[0].to_dict()
        print(f"   Using MLP Config: {best_config}")
    except:
        print("❌ Error: 'best_mlp_config.csv' not found. Run Stage 1 first.")
        return

    print("3. Extracting Features with MLP...")
    mlp = DynamicFusionMLP(
        X_concat.shape[1], 
        int(best_config['hidden_layers']), 
        int(best_config['fusion_dim']), 
        best_config['dropout']
    ).to(device)
    
    model_path = os.path.join(MODEL_DIR, 'Best_MLP_Stage1.pth')
    mlp.load_state_dict(torch.load(model_path))
    mlp.eval()
    
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_concat).to(device)
        fused_features_gpu = mlp(X_tensor)
        X_fused_cpu = fused_features_gpu.cpu().numpy()
        
    np.save(os.path.join(DATA_DIR, 'X_fused_hybrid_2.0.npy'), X_fused_cpu)

    print(f"\n4. Starting Grid Search on {X_fused_cpu.shape} features...")
    
    xgb = XGBClassifier(
        use_label_encoder=False, 
        eval_metric='logloss', 
        random_state=42, 
        tree_method='hist',
        device='cuda'
    )
    
    cv_splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42).split(X_fused_cpu, groups)
    
    grid = GridSearchCV(
        estimator=xgb,
        param_grid=XGB_PARAM_GRID,
        cv=cv_splitter,
        scoring='f1',
        verbose=1,
        n_jobs=4
    )

    grid.fit(X_fused_cpu, y)
    
    print("\n" + "-"*60)
    print(f"STAGE 2 result")
    print("-" * 60)
    print(f"   Best F1 Score: {grid.best_score_:.4f}")
    print(f"   Best Params: {grid.best_params_}")
    
    best_xgb = grid.best_estimator_
    with open(os.path.join(MODEL_DIR, 'Final_Hybrid_XGBoost.pkl'), 'wb') as f:
        pickle.dump(best_xgb, f)
        
    results = pd.DataFrame([grid.best_params_])
    results['best_f1_score'] = grid.best_score_
    results.to_csv(os.path.join(RESULTS_DIR, 'Stage2_XGBoost_Results.csv'), index=False)
    
    print(f"Stage 2 Complete. Final system ready in '{BASE_DIR}'")

if __name__ == "__main__":
    stage2_tune_xgboost()


STAGE 2: XGBOOST OPTIMIZATION
1. Loading Data...
2. Loading Best MLP Config...
   Using MLP Config: {'hidden_layers': 1.0, 'fusion_dim': 64.0, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.001}
3. Extracting Features with MLP...

4. Starting Grid Search on (1360, 64) features...
Fitting 10 folds for each of 240 candidates, totalling 2400 fits


C:\Users\User\Desktop\FYP_Coding\fyp_venv\lib\site-packages\xgboost\training.py:199: UserWarning: [08:38:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



------------------------------------------------------------
STAGE 2 result
------------------------------------------------------------
   Best F1 Score: 0.9833
   Best Params: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 400, 'subsample': 1.0}
Stage 2 Complete. Final system ready in 'Hybrid Model 2.0'


# Testing Result

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import pickle
import torch
import torch.nn as nn
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score


DATA_DIR = 'processed_data'
BASE_DIR = 'Hybrid Model 2.0'
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DynamicFusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, fusion_dim, dropout_rate):
        super(DynamicFusionMLP, self).__init__()
        self.layers = nn.ModuleList()
 
        self.layers.append(nn.Linear(input_dim, 256))
        self.layers.append(nn.BatchNorm1d(256))
        self.layers.append(nn.ReLU())
        self.layers.append(nn.Dropout(dropout_rate))
        
        current_dim = 256

        if hidden_layers == 2:
            self.layers.append(nn.Linear(256, 128))
            self.layers.append(nn.BatchNorm1d(128))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(dropout_rate))
            current_dim = 128

        self.fusion_layer = nn.Linear(current_dim, fusion_dim)
        self.fusion_act = nn.ReLU()
 
        self.head = nn.Linear(fusion_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        fused_features = self.fusion_act(self.fusion_layer(x))
        return fused_features

def evaluate_hybrid_v2():
    print(f"\n" + "="*60)
    print(f" HYBRID MODEL 2.0 TESTING")
    print("="*60)
    

    print("1. Loading Test Set...")
    try:
        X_absa_test = np.load(os.path.join(DATA_DIR, 'X_absa_test.npy'))
        X_emo_test  = np.load(os.path.join(DATA_DIR, 'X_emo_test.npy'))
        y_test      = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    except FileNotFoundError:
        print("Error: Test files not found. Check 'processed_data' folder.")
        return

    X_concat_test = np.hstack([X_absa_test, X_emo_test])
    print(f"   Test Set Shape: {X_concat_test.shape}")


    print("\n2. Loading Best Configuration...")
    try:
        config_path = os.path.join(RESULTS_DIR, 'best_mlp_config.csv')
        config_df = pd.read_csv(config_path)
        best_config = config_df.iloc[0].to_dict()
        print(f"   Detected Config: {best_config}")
    except FileNotFoundError:
        print("Error: 'best_mlp_config.csv' not found. Did you run Stage 1?")
        return


    print("\n3. Loading MLP Model...")
    mlp = DynamicFusionMLP(
        input_dim=X_concat_test.shape[1],
        hidden_layers=int(best_config['hidden_layers']),
        fusion_dim=int(best_config['fusion_dim']),
        dropout_rate=best_config['dropout']
    ).to(device)
    
    model_path = os.path.join(MODEL_DIR, 'Best_MLP_Stage1.pth')
    try:
        mlp.load_state_dict(torch.load(model_path))
        mlp.eval()
        for param in mlp.parameters():
            param.requires_grad = False
        print("   ✅ MLP Loaded Successfully")
    except Exception as e:
        print(f"Error loading MLP: {e}")
        return

    print("\n4. Generating Fused Features for Test Data...")
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_concat_test).to(device)
        features_gpu = mlp(X_tensor)
        X_test_fused = features_gpu.cpu().numpy()
        
    print("\n5. Loading XGBoost Model...")
    xgb_path = os.path.join(MODEL_DIR, 'Final_Hybrid_XGBoost.pkl')
    try:
        with open(xgb_path, 'rb') as f:
            xgb = pickle.load(f)
        print("   ✅ XGBoost Loaded Successfully")
    except Exception as e:
        print(f"Error loading XGBoost: {e}")
        return

    print("\n6. Predicting...")
    y_pred = xgb.predict(X_test_fused)
    
    print("\n" + "="*60)
    print("HYBRID 2.0 TEST RESULTS")
    print("="*60)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"   Accuracy: {acc:.4f}")
    print(f"   F1 Score: {f1:.4f}")
    print("\n   --- Classification Report ---")
    print(classification_report(y_test, y_pred, digits=4))
    
    print("   --- Confusion Matrix ---")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

if __name__ == "__main__":
    evaluate_hybrid_v2()


 HYBRID MODEL 2.0 TESTING
1. Loading Test Set...
   Test Set Shape: (240, 1536)

2. Loading Best Configuration...
   Detected Config: {'hidden_layers': 1.0, 'fusion_dim': 64.0, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.001}

3. Loading MLP Model...
   ✅ MLP Loaded Successfully

4. Generating Fused Features for Test Data...

5. Loading XGBoost Model...
   ✅ XGBoost Loaded Successfully

6. Predicting...

HYBRID 2.0 TEST RESULTS
   Accuracy: 0.8625
   F1 Score: 0.8631

   --- Classification Report ---
              precision    recall  f1-score   support

           0     0.8655    0.8583    0.8619       120
           1     0.8595    0.8667    0.8631       120

    accuracy                         0.8625       240
   macro avg     0.8625    0.8625    0.8625       240
weighted avg     0.8625    0.8625    0.8625       240

   --- Confusion Matrix ---
[[103  17]
 [ 16 104]]
